In [ ]:
!pip install buildozer

In [ ]:
!pip install cython==0.29.33

In [ ]:
!sudo apt-get update
!sudo apt install -y git zip unzip openjdk-17-jdk python3-pip autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo5 cmake libffi-dev libssl-dev --fix-missing

In [ ]:
!buildozer init   #say yes. Also edit your buildozer.spec file like i did(you can find prepared one on my Github)

In [ ]:
!sed -i 's/warn_on_root = 1/warn_on_root = 0/' buildozer.spec
!sed -i 's/.*android.accept_sdk_license.*/android.accept_sdk_license = True/' buildozer.spec
!buildozer android debug 2>&1 | tail -300    #compile to apk

In [ ]:
import glob        #this for patching Sympy use it after 1st compile then add 'mpmath' in buildozer.spec file
import os           #then run it this section. then use previous compile section again.

# patching sympy for arm64-v8a
base_dir = "/content/.buildozer/android/platform/build-*/build/python-installs/*/arm64-v8a/sympy/**/*.py"

file_paths = glob.glob(base_dir, recursive=True)

patched_files_count = 0

for path in file_paths:
    try:
        with open(path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        modified = False
        for i in range(len(lines)):
            line = lines[i]

            if '__doc__' in line and ('=' in line):
                indent = line[:len(line) - len(line.lstrip('\t '))]

                if i > 0 and 'try:' in lines[i-1]:
                    continue
                stripped_line = line.lstrip()
                if not stripped_line.startswith('#') and '==' not in stripped_line:

                    safe_block = f"{indent}try:\n{indent}    {stripped_line}{indent}except Exception:\n{indent}    pass\n"

                    lines[i] = safe_block
                    modified = True

        if modified:
            with open(path, 'w', encoding='utf-8') as f:
                f.writelines(lines)
            print(f"PATCHED: {path}")
            patched_files_count += 1

    except Exception as e:
        print(f"ERROR ({path}): {e}")

print(f"\nTOTAL {patched_files_count} FILE PATCHED")